# USAD — Transfer Learning Plan A
## Multitask: Encoder Compartido + Decoders por Estación SIATA

**Estrategia:** Transfer learning del modelo USAD preentrenado (SWaT dataset, 51 features) hacia
4 estaciones meteorológicas SIATA de Medellín (temperatura, Plan A).

**Arquitectura:** Un encoder compartido que capta correlaciones cruzadas entre estaciones + 4 pares de decoders especializados, uno por estación.

**Transferibilidad:** Las capas internas (`linear2`, `linear3` del encoder; `linear1`, `linear2` de cada decoder) tienen exactamente las mismas dimensiones que el modelo preentrenado (h1=306, h2=153, z=1200). Solo se re-inicializan la primera y última capa.

| Split | Período | Uso |
|-------|---------|-----|
| **E** | Mar–Abr 2023 | Entrenamiento (~44.8%) |
| **V** | Finales Abr 2023 | Validación + calibración umbral (~8.6%) |
| **T** | Jun–Jul 2023 | Test — estrés, alta concentración anomalías 203-UNAL (~46.6%) |

In [ ]:
import subprocess, sys
for pkg in ['seaborn']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import io
import warnings

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    roc_curve, roc_auc_score, f1_score,
    precision_score, recall_score
)

warnings.filterwarnings('ignore')
print(f'PyTorch: {torch.__version__}')
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Configuración
> **⚠️ Actualiza `GITHUB_USER`, `GITHUB_REPO` y `GITHUB_BRANCH` con tu información.**

In [ ]:
# ── GitHub ─────────────────────────────────────────────────────────────────
GITHUB_USER   = 'TU_USUARIO'            # <- Actualizar
GITHUB_REPO   = 'monografia'             # <- Actualizar
GITHUB_BRANCH = 'feature/transfer-learning'

_BASE      = f'https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}'
DATA_URL   = f'{_BASE}/modelos/usad/data/plan_a/'
MODEL_URL  = f'{_BASE}/modelos/usad/model.pth'

# ── Arquitectura ────────────────────────────────────────────────────────────
STATION_IDS  = ['68', '201', '203', '478']
WINDOW_SIZE  = 12          # timesteps por ventana
N_STATIONS   = 4
W_SIZE       = WINDOW_SIZE * N_STATIONS   # 48
LATENT_SIZE  = 1200        # mismo que SWaT (z_size = 12 * 100)
H1_ENC       = 306         # int(612 / 2) — retenido del modelo preentrenado
H2_ENC       = 153         # int(612 / 4) — retenido del modelo preentrenado
H1_DEC       = 153         # int(612 / 4) — retenido del modelo preentrenado
H2_DEC       = 306         # int(612 / 2) — retenido del modelo preentrenado

# ── Entrenamiento ───────────────────────────────────────────────────────────
BATCH_SIZE = 256
N_EPOCHS   = 100
LR         = 1e-4

# Indices de cada estacion dentro de la ventana de 48 dims (layout contiguo)
# Ventana = [s68_t0..t11 | s201_t0..t11 | s203_t0..t11 | s478_t0..t11]
STATION_SLICES = {
    sid: slice(i * WINDOW_SIZE, (i + 1) * WINDOW_SIZE)
    for i, sid in enumerate(STATION_IDS)
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
STATION_NAMES = {'68': '68-Jardín Botánico', '201': '201-Torre SIATA',
                 '203': '203-UNAL', '478': '478-Fiscalía'}

print(f'Device: {DEVICE}')
print(f'W_SIZE={W_SIZE}, LATENT_SIZE={LATENT_SIZE}')
print(f'Station slices: {STATION_SLICES}')

## Carga de Datos
Se cargan los 4 CSV directamente desde GitHub (sin archivos locales).

In [ ]:
def fetch_csv(url: str) -> pd.DataFrame:
    resp = requests.get(url)
    resp.raise_for_status()
    return pd.read_csv(io.StringIO(resp.text), parse_dates=['fecha_hora'])


raw = {}
for sid in STATION_IDS:
    url = DATA_URL + f'{sid}.csv'
    raw[sid] = fetch_csv(url)
    print(f'Estacion {sid}: {len(raw[sid]):,} filas | splits: {raw[sid]["Split"].value_counts().to_dict()}')

# Merge de las 4 estaciones por fecha_hora (inner join)
merged = raw[STATION_IDS[0]][['fecha_hora', 'Split']].copy()
for sid in STATION_IDS:
    subset = raw[sid][['fecha_hora', 't', 'flag']].rename(
        columns={'t': f't_{sid}', 'flag': f'flag_{sid}'}
    )
    merged = merged.merge(subset, on='fecha_hora', how='inner')

merged = merged.sort_values('fecha_hora').reset_index(drop=True)

# Flag combinado: anomalo si CUALQUIER estacion tiene flag=1
FLAG_COLS    = [f'flag_{sid}' for sid in STATION_IDS]
STATION_COLS = [f't_{sid}'    for sid in STATION_IDS]
merged['flag_any'] = (merged[FLAG_COLS] > 0).any(axis=1).astype(int)

print(f'\nMerged: {len(merged):,} filas comunes')
print(f'Splits:\n{merged["Split"].value_counts()}')
print(f'\nAnomalias en Split=T: {merged[merged["Split"]=="T"]["flag_any"].sum():,}')
for sid in STATION_IDS:
    n = merged[merged['Split']=='T'][f'flag_{sid}'].sum()
    print(f'  Estacion {sid}: {n:,} puntos anomalos en test')
merged.head(3)

## Arquitectura — Principios SOLID

| Clase | SOLID | Responsabilidad |
|-------|-------|----------------|
| `TransferEncoder` | S | Codificacion con dims compatibles con SWaT |
| `TransferDecoder` | S | Decodificacion con dims compatibles con SWaT |
| `StationDecoderPair` | S | Par (decoder1, decoder2) de una sola estacion |
| `MultitaskUsadModel` | S,O,I,D | Orquestacion multitask; extensible |
| `WeightTransferService` | S,O | Transferencia de pesos desde `model.pth` |
| `SIATADataset` | S | Dataset por split con soporte de labels |
| `AnomalyEvaluator` | S | Calculo de metricas de deteccion |

In [ ]:
class TransferEncoder(nn.Module):
    """
    Encoder con dimensiones internas retenidas del preentrenado SWaT.
    Capas transferibles  : linear2 (306->153), linear3 (153->1200)
    Capas re-inicializadas: linear1 (in_size->306) — input cambia de 612 a 48
    """
    def __init__(self, in_size: int, latent_size: int,
                 h1: int = H1_ENC, h2: int = H2_ENC):
        super().__init__()
        self.linear1 = nn.Linear(in_size, h1)       # re-inicializada
        self.linear2 = nn.Linear(h1, h2)             # transferible
        self.linear3 = nn.Linear(h2, latent_size)    # transferible

    def forward(self, w: torch.Tensor) -> torch.Tensor:
        return F.relu(self.linear3(F.relu(self.linear2(F.relu(self.linear1(w))))))


class TransferDecoder(nn.Module):
    """
    Decoder con dimensiones internas retenidas del preentrenado SWaT.
    Capas transferibles  : linear1 (1200->153), linear2 (153->306)
    Capas re-inicializadas: linear3 (306->out_size) — output cambia de 612 a 48
    """
    def __init__(self, latent_size: int, out_size: int,
                 h1: int = H1_DEC, h2: int = H2_DEC):
        super().__init__()
        self.linear1 = nn.Linear(latent_size, h1)    # transferible
        self.linear2 = nn.Linear(h1, h2)             # transferible
        self.linear3 = nn.Linear(h2, out_size)       # re-inicializada

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.linear3(F.relu(self.linear2(F.relu(self.linear1(z))))))


class StationDecoderPair(nn.Module):
    """Par (decoder1, decoder2) de una sola estacion. (Single Responsibility)"""
    def __init__(self, latent_size: int, out_size: int):
        super().__init__()
        self.decoder1 = TransferDecoder(latent_size, out_size)
        self.decoder2 = TransferDecoder(latent_size, out_size)


# Verificar dimensiones
_enc = TransferEncoder(W_SIZE, LATENT_SIZE)
print('TransferEncoder(in=48, latent=1200):')
for name, layer in _enc.named_children():
    transferable = name in ['linear2', 'linear3']
    tag = 'TRANSFERIBLE' if transferable else 're-inicializada'
    print(f'  {name}: {layer.in_features} -> {layer.out_features}  [{tag}]')
del _enc

In [ ]:
class MultitaskUsadModel(nn.Module):
    """
    USAD Multitask: encoder compartido + pares de decoders por estacion.

    SOLID:
    - S: orquesta forward pass y losses; no gestiona datos ni metricas.
    - O: agregar estaciones pasando mas IDs al constructor, sin modificar la clase.
    - I: training_step separado de anomaly_score (entrenamiento vs inferencia).
    - D: encoder inyectado por constructor (no instanciado internamente).
    """
    def __init__(self, encoder: TransferEncoder,
                 station_ids: list, w_size: int, latent_size: int):
        super().__init__()
        self.encoder = encoder
        self.w_size   = w_size
        self.station_decoders = nn.ModuleDict({
            sid: StationDecoderPair(latent_size, w_size)
            for sid in station_ids
        })

    # ── Interfaz de entrenamiento ────────────────────────────────────────
    def training_step(self, batch: torch.Tensor, station_id: str, n: int):
        """Devuelve (loss1, loss2) para una estacion y un epoch n."""
        pair = self.station_decoders[station_id]
        z    = self.encoder(batch)
        w1   = pair.decoder1(z)
        w2   = pair.decoder2(z)
        w3   = pair.decoder2(self.encoder(w1))   # reconstruccion adversarial
        loss1 = (1/n) * torch.mean((batch-w1)**2) + (1-1/n) * torch.mean((batch-w3)**2)
        loss2 = (1/n) * torch.mean((batch-w2)**2) - (1-1/n) * torch.mean((batch-w3)**2)
        return loss1, loss2

    def validation_step(self, batch: torch.Tensor, station_id: str, n: int):
        with torch.no_grad():
            return self.training_step(batch, station_id, n)

    # ── Interfaz de inferencia (Interface Segregation) ───────────────────
    @torch.no_grad()
    def anomaly_score(self, batch: torch.Tensor, station_id: str,
                      alpha: float = 0.5, beta: float = 0.5) -> torch.Tensor:
        """
        Score de anomalia por ventana para una estacion especifica.
        Formula original USAD: alpha*MSE(batch,w1) + beta*MSE(batch,w2)
        donde w2 = decoder2(encoder(w1)) — igual que testing() en usad.py.
        MSE calculado sobre el slice de 12 dims de la estacion.
        """
        pair = self.station_decoders[station_id]
        z    = self.encoder(batch)
        w1   = pair.decoder1(z)
        w2   = pair.decoder2(self.encoder(w1))
        s    = STATION_SLICES[station_id]
        return (alpha * torch.mean((batch[:, s] - w1[:, s])**2, dim=1) +
                beta  * torch.mean((batch[:, s] - w2[:, s])**2, dim=1))


# Resumen de parametros
_enc   = TransferEncoder(W_SIZE, LATENT_SIZE)
_model = MultitaskUsadModel(_enc, STATION_IDS, W_SIZE, LATENT_SIZE)
total  = sum(p.numel() for p in _model.parameters())
enc_tr = sum(
    getattr(_model.encoder, l).weight.numel() + getattr(_model.encoder, l).bias.numel()
    for l in ['linear2', 'linear3']
)
dec_tr_per = sum(
    getattr(_model.station_decoders['68'].decoder1, l).weight.numel() +
    getattr(_model.station_decoders['68'].decoder1, l).bias.numel()
    for l in ['linear1', 'linear2']
)
transferable = enc_tr + 2 * N_STATIONS * dec_tr_per
print(f'Parametros totales      : {total:,}')
print(f'Parametros transferibles: {transferable:,} ({transferable/total*100:.1f}%)')
print(f'Parametros nuevos       : {total-transferable:,} ({(total-transferable)/total*100:.1f}%)')
del _enc, _model

In [ ]:
class WeightTransferService:
    """
    Carga y transfiere pesos compatibles desde el checkpoint SWaT.
    SOLID — S: unica responsabilidad: transferencia de pesos.
    SOLID — O: extensible para nuevas estrategias sin modificar la clase.
    """
    ENCODER_LAYERS = ['linear2', 'linear3']
    DECODER_LAYERS = ['linear1', 'linear2']

    @staticmethod
    def load_checkpoint(url: str) -> dict:
        print('Descargando model.pth desde GitHub...')
        resp = requests.get(url)
        resp.raise_for_status()
        state = torch.load(io.BytesIO(resp.content), map_location='cpu')
        print(f'  Checkpoint cargado: {len(state)} tensores')
        return state

    @classmethod
    def transfer_to_model(cls, model: MultitaskUsadModel,
                          state_dict: dict, verbose: bool = True) -> None:
        """
        Copia pesos compatibles al modelo.
        encoder.linear2/3 y decoder{1,2}.linear1/2 por cada estacion.
        """
        # Encoder: linear2 y linear3
        for lname in cls.ENCODER_LAYERS:
            layer = getattr(model.encoder, lname)
            layer.weight.data.copy_(state_dict[f'encoder.{lname}.weight'])
            layer.bias.data.copy_(state_dict[f'encoder.{lname}.bias'])

        # Decoders de cada estacion: linear1 y linear2
        # Ambos (decoder1/decoder2) parten de los mismos pesos SWaT y divergen
        for sid in model.station_decoders:
            pair = model.station_decoders[sid]
            for dec_attr, swat_pfx in [('decoder1', 'decoder1'), ('decoder2', 'decoder2')]:
                dec = getattr(pair, dec_attr)
                for lname in cls.DECODER_LAYERS:
                    layer = getattr(dec, lname)
                    layer.weight.data.copy_(state_dict[f'{swat_pfx}.{lname}.weight'])
                    layer.bias.data.copy_(state_dict[f'{swat_pfx}.{lname}.bias'])

        if verbose:
            enc_params = sum(
                state_dict[f'encoder.{l}.weight'].numel() + state_dict[f'encoder.{l}.bias'].numel()
                for l in cls.ENCODER_LAYERS
            )
            dec_params = sum(
                state_dict[f'decoder1.{l}.weight'].numel() + state_dict[f'decoder1.{l}.bias'].numel()
                for l in cls.DECODER_LAYERS
            )
            total_transferred = enc_params + 2 * len(model.station_decoders) * dec_params
            total_params = sum(p.numel() for p in model.parameters())
            print('Transfer completado:')
            print(f'  encoder.linear2/linear3           : {enc_params:,} params')
            print(f'  decoder{{1,2}}.linear1/2 x {len(model.station_decoders)} stations: {2*len(model.station_decoders)*dec_params:,} params')
            print(f'  Total transferido : {total_transferred:,} / {total_params:,} ({total_transferred/total_params*100:.1f}%)')


print('WeightTransferService definido.')

In [ ]:
class SIATADataset(Dataset):
    """
    Dataset multivariado de 4 estaciones SIATA para un split especifico.
    Layout de ventanas (contiguo por estacion, 48 dims):
      [t68_0..t68_11 | t201_0..t201_11 | t203_0..t203_11 | t478_0..t478_11]

    SOLID — S: encapsula un unico split con su normalizacion.
    """
    def __init__(self, df: pd.DataFrame, split: str,
                 window_size: int, station_cols: list,
                 scaler: MinMaxScaler = None, flag_col: str = None):
        data_split = df[df['Split'] == split].copy()
        values = data_split[station_cols].values.astype('float32')

        # MinMaxScaler: fit SOLO en Split=E para evitar data leakage
        if scaler is None:
            self.scaler = MinMaxScaler()
            values = self.scaler.fit_transform(values)
        else:
            self.scaler = scaler
            values = scaler.transform(values)

        n = len(values) - window_size
        # (window_size, n_stations).T.flatten() -> layout contiguo por estacion
        self.windows = torch.from_numpy(
            np.stack([values[i:i+window_size].T.flatten() for i in range(n)])
        )

        self.labels = None
        if flag_col and flag_col in data_split.columns:
            flags = data_split[flag_col].values
            self.labels = torch.tensor(
                [int(np.any(flags[i:i+window_size] > 0)) for i in range(n)],
                dtype=torch.long
            )

    def __len__(self): return len(self.windows)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.windows[idx], self.labels[idx]
        return self.windows[idx]


class AnomalyEvaluator:
    """
    Calcula metricas de deteccion de anomalias.
    SOLID — S: unica responsabilidad: evaluacion de metricas.
    """
    @staticmethod
    def compute_scores(model, loader, station_id, device):
        model.eval()
        scores, labels = [], []
        for batch, label in loader:
            s = model.anomaly_score(batch.to(device), station_id)
            scores.extend(s.cpu().numpy())
            labels.extend(label.numpy())
        return np.array(scores), np.array(labels)

    @staticmethod
    def find_threshold(scores, labels) -> float:
        """Umbral optimo por maximo F1 en el set de validacion."""
        _, _, thresholds = roc_curve(labels, scores)
        best_t, best_f1 = thresholds[0], 0.0
        for t in thresholds:
            f1 = f1_score(labels, scores >= t, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        return float(best_t)

    @staticmethod
    def report(scores, labels, threshold, station_id='') -> dict:
        preds = (scores >= threshold).astype(int)
        metrics = {
            'station'  : station_id,
            'roc_auc'  : roc_auc_score(labels, scores) if len(np.unique(labels)) > 1 else 0.0,
            'f1'       : f1_score(labels, preds, zero_division=0),
            'precision': precision_score(labels, preds, zero_division=0),
            'recall'   : recall_score(labels, preds, zero_division=0),
            'threshold': threshold
        }
        print(f'[Estacion {station_id}] ROC-AUC={metrics["roc_auc"]:.4f} | '
              f'F1={metrics["f1"]:.4f} | P={metrics["precision"]:.4f} | '
              f'R={metrics["recall"]:.4f} | umbral={threshold:.6f}')
        return metrics


print('SIATADataset y AnomalyEvaluator definidos.')

## Preprocesamiento

El `MinMaxScaler` se ajusta **unicamente** sobre `Split=E`.
La misma transformacion se aplica a `Split=V` y `Split=T` para evitar data leakage.

In [ ]:
# Entrenamiento — scaler ajustado en Split=E
train_ds = SIATADataset(merged, 'E', WINDOW_SIZE, STATION_COLS)
scaler   = train_ds.scaler   # reutilizar en V y T

# Validacion — transform con scaler de E, incluye labels
val_ds   = SIATADataset(merged, 'V', WINDOW_SIZE, STATION_COLS,
                        scaler=scaler, flag_col='flag_any')

# Test — transform con scaler de E, labels globales y por estacion 203
test_ds     = SIATADataset(merged, 'T', WINDOW_SIZE, STATION_COLS,
                           scaler=scaler, flag_col='flag_any')
test_203_ds = SIATADataset(merged, 'T', WINDOW_SIZE, STATION_COLS,
                           scaler=scaler, flag_col='flag_203')

# DataLoaders
train_loader    = DataLoader(train_ds,     batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader      = DataLoader(val_ds,       batch_size=BATCH_SIZE, shuffle=False)
test_loader     = DataLoader(test_ds,      batch_size=BATCH_SIZE, shuffle=False)
test_203_loader = DataLoader(test_203_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train : {len(train_ds):,} ventanas ({len(train_loader)} batches)')
print(f'Val   : {len(val_ds):,} ventanas  | anomalas: {val_ds.labels.sum().item() if val_ds.labels is not None else "N/A"}')
print(f'Test  : {len(test_ds):,} ventanas | anomalas: {test_ds.labels.sum().item() if test_ds.labels is not None else "N/A"}')
print(f'Test-203: {test_203_ds.labels.sum().item()} ventanas anomalas (solo estacion 203-UNAL)')
print(f'Forma ventana: {train_ds[0].shape}')

## Transfer Learning — Carga del Modelo Preentrenado

Se instancia `MultitaskUsadModel` y se cargan los pesos compatibles desde `model.pth`.

In [ ]:
# 1. Cargar checkpoint SWaT desde GitHub
state_dict = WeightTransferService.load_checkpoint(MODEL_URL)

# 2. Instanciar encoder (Dependency Inversion — inyectado en el modelo)
encoder = TransferEncoder(in_size=W_SIZE, latent_size=LATENT_SIZE)

# 3. Instanciar modelo multitask (Open/Closed)
model = MultitaskUsadModel(
    encoder=encoder,
    station_ids=STATION_IDS,
    w_size=W_SIZE,
    latent_size=LATENT_SIZE
).to(DEVICE)

# 4. Transferir pesos
WeightTransferService.transfer_to_model(model, state_dict, verbose=True)

# Sanity check: val_loss antes de entrenar (con pesos transferidos)
model.eval()
pre_losses = {sid: [] for sid in STATION_IDS}
with torch.no_grad():
    for batch, _ in val_loader:
        batch = batch.to(DEVICE)
        for sid in STATION_IDS:
            l1, _ = model.validation_step(batch, sid, n=1)
            pre_losses[sid].append(l1.item())

print('\nVal loss ANTES del entrenamiento (con pesos SWaT transferidos):')
for sid in STATION_IDS:
    print(f'  Estacion {sid}: loss1={np.mean(pre_losses[sid]):.4f}')

## Entrenamiento — Fine-tuning Multitask

Loop siguiendo el patron original de USAD: **dos pasos de optimizacion por batch**,
uno para `optimizer1` (encoder + decoder1) y otro para `optimizer2` (encoder + decoder2).

In [ ]:
def build_optimizers(model: MultitaskUsadModel, lr: float = LR) -> dict:
    opts = {}
    for sid, pair in model.station_decoders.items():
        opts[sid] = {
            'opt1': Adam(list(model.encoder.parameters()) + list(pair.decoder1.parameters()), lr=lr),
            'opt2': Adam(list(model.encoder.parameters()) + list(pair.decoder2.parameters()), lr=lr)
        }
    return opts

opts = build_optimizers(model)
print(f'{len(opts)} pares de optimizadores creados (uno por estacion).')

In [ ]:
history = {sid: {'train_l1': [], 'train_l2': [], 'val_l1': [], 'val_l2': []}
           for sid in STATION_IDS}

for epoch in range(1, N_EPOCHS + 1):

    # Entrenamiento
    model.train()
    ep_losses = {sid: {'l1': 0.0, 'l2': 0.0, 'n': 0} for sid in STATION_IDS}

    for batch in train_loader:
        batch = batch.to(DEVICE)
        for sid in STATION_IDS:
            # Paso 1: encoder + decoder1 (patron original USAD)
            loss1, _ = model.training_step(batch, sid, epoch)
            opts[sid]['opt1'].zero_grad()
            loss1.backward()
            opts[sid]['opt1'].step()
            # Paso 2: encoder + decoder2
            _, loss2 = model.training_step(batch, sid, epoch)
            opts[sid]['opt2'].zero_grad()
            loss2.backward()
            opts[sid]['opt2'].step()

            ep_losses[sid]['l1'] += loss1.item()
            ep_losses[sid]['l2'] += loss2.item()
            ep_losses[sid]['n']  += 1

    # Validacion
    model.eval()
    val_losses = {sid: {'l1': 0.0, 'l2': 0.0, 'n': 0} for sid in STATION_IDS}
    with torch.no_grad():
        for batch, _ in val_loader:
            batch = batch.to(DEVICE)
            for sid in STATION_IDS:
                l1, l2 = model.validation_step(batch, sid, epoch)
                val_losses[sid]['l1'] += l1.item()
                val_losses[sid]['l2'] += l2.item()
                val_losses[sid]['n']  += 1

    for sid in STATION_IDS:
        history[sid]['train_l1'].append(ep_losses[sid]['l1']  / ep_losses[sid]['n'])
        history[sid]['train_l2'].append(ep_losses[sid]['l2']  / ep_losses[sid]['n'])
        history[sid]['val_l1'].append(val_losses[sid]['l1'] / val_losses[sid]['n'])
        history[sid]['val_l2'].append(val_losses[sid]['l2'] / val_losses[sid]['n'])

    if epoch % 10 == 0 or epoch == 1:
        summary = ' | '.join(f'st{sid} vl1={history[sid]["val_l1"][-1]:.4f}'
                             for sid in STATION_IDS)
        print(f'Epoch {epoch:3d}/{N_EPOCHS} — {summary}')

print('\nEntrenamiento completado.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, sid in zip(axes, STATION_IDS):
    ep = range(1, N_EPOCHS + 1)
    ax.plot(ep, history[sid]['val_l1'], '-',  label='val_loss1', linewidth=1.5)
    ax.plot(ep, history[sid]['val_l2'], '--', label='val_loss2', linewidth=1.5)
    ax.set_title(STATION_NAMES[sid], fontsize=11)
    ax.set_xlabel('Epoca')
    ax.set_ylabel('Loss (MSE)')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Curvas de Validacion — USAD Multitask Transfer Learning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Evaluacion

1. Umbral optimo buscado en `Split=V` (maximiza F1, sin tocar datos de test)
2. Metricas sobre `Split=T`

In [ ]:
thresholds = {}
for sid in STATION_IDS:
    val_scores, val_labels = AnomalyEvaluator.compute_scores(model, val_loader, sid, DEVICE)
    if val_labels.sum() == 0:
        thresholds[sid] = float(np.percentile(val_scores, 95))
        print(f'Estacion {sid}: sin anomalias en val -> umbral P95 = {thresholds[sid]:.6f}')
    else:
        thresholds[sid] = AnomalyEvaluator.find_threshold(val_scores, val_labels)
        print(f'Estacion {sid}: umbral max-F1 en val = {thresholds[sid]:.6f}')

In [ ]:
results = {}
for sid in STATION_IDS:
    test_scores, test_labels = AnomalyEvaluator.compute_scores(model, test_loader, sid, DEVICE)
    results[sid] = AnomalyEvaluator.report(test_scores, test_labels, thresholds[sid], station_id=sid)
    results[sid]['scores'] = test_scores
    results[sid]['labels'] = test_labels

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, sid in zip(axes, STATION_IDS):
    scores = results[sid]['scores']
    labels = results[sid]['labels']
    if len(np.unique(labels)) < 2:
        ax.set_title(f'{STATION_NAMES[sid]}\n(sin anomalias en test)')
        continue
    fpr, tpr, _ = roc_curve(labels, scores)
    auc = roc_auc_score(labels, scores)
    ax.plot(fpr, tpr, label=f'AUC = {auc:.4f}', linewidth=2)
    ax.plot([0, 1], [1, 0], 'r:', linewidth=1)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'ROC — {STATION_NAMES[sid]}', fontsize=11)
    ax.legend(loc='lower right'); ax.grid(alpha=0.3)

plt.suptitle('Curvas ROC por Estacion — Split T (Test)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(STATION_IDS), figsize=(16, 4))
for ax, sid in zip(axes, STATION_IDS):
    preds = (results[sid]['scores'] >= thresholds[sid]).astype(int)
    cm = pd.crosstab(
        pd.Series(preds,                 name='Predicted'),
        pd.Series(results[sid]['labels'], name='Actual')
    )
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f"{STATION_NAMES[sid]}\nF1={results[sid]['f1']:.3f}", fontsize=10)
plt.suptitle('Matrices de Confusion — Split T', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Analisis Estacion 203-UNAL

Escenario de estres principal del Plan A: alta concentracion de anomalias en Jun–Jul 2023.

In [ ]:
scores_203, labels_203 = AnomalyEvaluator.compute_scores(model, test_203_loader, '203', DEVICE)
metrics_203 = AnomalyEvaluator.report(scores_203, labels_203, thresholds['203'],
                                        station_id='203 (labels propios)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Histograma: normal vs anomalo
ax1.hist([scores_203[labels_203 == 0], scores_203[labels_203 == 1]],
         bins=50, color=['#82E0AA', '#EC7063'],
         label=['Normal', 'Anomalia'], stacked=True, alpha=0.8)
ax1.axvline(thresholds['203'], color='navy', linestyle='--',
            label=f'Umbral = {thresholds["203"]:.5f}')
ax1.set_title('Distribucion de Scores — 203-UNAL', fontsize=12)
ax1.set_xlabel('Anomaly Score'); ax1.set_ylabel('Frecuencia')
ax1.legend(); ax1.grid(alpha=0.3)

# Serie temporal
test_times = merged[merged['Split'] == 'T']['fecha_hora'].values[WINDOW_SIZE:]
n_plot = min(len(test_times), len(scores_203))
ax2.plot(test_times[:n_plot], scores_203[:n_plot], alpha=0.6, linewidth=0.5, label='Anomaly score')
ax2.axhline(thresholds['203'], color='navy', linestyle='--', linewidth=1, label='Umbral')
anom_mask = labels_203[:n_plot] == 1
if anom_mask.any():
    y_top = max(scores_203[:n_plot]) * 1.05
    ax2.scatter(test_times[:n_plot][anom_mask],
                [y_top] * anom_mask.sum(),
                color='red', s=5, alpha=0.5, label='Anomalia real', zorder=5)
ax2.set_title('Serie Temporal de Scores — 203-UNAL (Split T)', fontsize=12)
ax2.set_xlabel('Fecha'); ax2.set_ylabel('Score')
ax2.legend(); ax2.grid(alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Resumen de Resultados

In [ ]:
rows = []
for sid in STATION_IDS:
    r = results[sid]
    rows.append({
        'Estacion'  : STATION_NAMES[sid],
        'ROC-AUC'   : f"{r['roc_auc']:.4f}",
        'F1'        : f"{r['f1']:.4f}",
        'Precision' : f"{r['precision']:.4f}",
        'Recall'    : f"{r['recall']:.4f}",
        'Umbral'    : f"{r['threshold']:.6f}"
    })

df_results = pd.DataFrame(rows).set_index('Estacion')
print('=' * 72)
print('RESULTADOS — USAD Transfer Learning Multitask (Plan A)')
print('=' * 72)
print(df_results.to_string())
print('=' * 72)
print(f'Modelo : MultitaskUsadModel | Encoder compartido + {N_STATIONS} pares de decoders')
print(f'Epocas : {N_EPOCHS} | LR: {LR} | Batch: {BATCH_SIZE} | W_SIZE: {W_SIZE}')
print(f'Pesos transferidos: encoder.linear2/3 + decoder{{1,2}}.linear1/2')
df_results